<!-- VIDEO: 1 / LangChain v1.0 — 왜 필요한가 + init_chat_model -->

# 02. LangChain Core + LCEL

> **학습 목표**
> 1. LangChain v1.0의 단순화된 네임스페이스(`langchain.messages`, `langchain.chat_models` 등)의 구조를 이해한다.
> 2. `init_chat_model`로 로컬 Ollama 모델을 초기화하고, 나중에 모델 식별자만 바꿔 다른 제공자로 전환하는 방법을 이해한다.
> 3. **Messages → Prompt Template → Output Parser** 의 표준 처리 흐름을 익힌다.
> 4. **LCEL**의 `|` 파이프 연산자로 체인을 선언적으로 구성한다.
> 5. `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`로 분기, 병렬 실행, 전후처리를 구현한다.
>
> **선수 지식**: 01번 노트북(LLM 호출 기본 개념).

---

> **LangChain v1.0의 변경점**
> - `langchain` 네임스페이스가 5개로 단순화됨: `langchain.messages`, `langchain.tools`, `langchain.agents`, `langchain.chat_models`, `langchain.embeddings`.
> - `LLMChain`, 전통 Retriever 등 레거시 기능은 `langchain-classic` 패키지로 분리됨.
> - 이 프로젝트는 v1.0 이상의 API만 사용합니다.

01번 노트북과 마찬가지로 지금은 로컬 Ollama의 `gemma4:e2b`를 사용합니다. 본 노트북부터는 `init_chat_model("ollama:gemma4:e2b")`로 모델을 초기화하고, 나중에 모델 식별자만 변경해 다른 제공자로 전환하는 방법을 학습합니다.

---

## 1. 환경 설정 — LLM 선택

지금은 API 키가 필요 없는 로컬 Ollama를 사용합니다. Gemini를 비롯한 다른 제공자는 나중에 사용할 수 있도록 선택지로만 남겨 둡니다.

```
[권장 순서]
현재: Ollama Gemma 4 E2B         (로컬 실행, API 키 불필요)
나중: Gemini 2.5 Flash-Lite     (Google API 키 필요)
나중: Groq Qwen3 32B            (Groq API 키 필요)
나중: OpenAI gpt-4.1-mini       (OpenAI API 키 필요)
```

In [2]:
from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings("ignore")

import os
print("✅ 키 감지:",
      "Gemini" if os.getenv("GOOGLE_API_KEY") else "—",
      "Groq" if os.getenv("GROQ_API_KEY") else "—",
      "OpenAI" if os.getenv("OPENAI_API_KEY") else "—")

✅ 키 감지: — — —


## 2. Models — `init_chat_model`

LangChain v1.0에서는 `init_chat_model("provider:model")` 한 함수로 모든 제공자의 모델을 통합 초기화할 수 있습니다. 모델 문자열만 변경하면 다른 제공자로 전환되며, 호출 인터페이스는 동일하게 유지됩니다.

In [3]:
from langchain.chat_models import init_chat_model

# 현재 사용: Ollama Gemma 4 E2B (로컬)
llm = init_chat_model("ollama:gemma4:e2b", temperature=0.3)

# 나중에 사용: Google Gemini 2.5 Flash-Lite
# llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0.3)

# 나중에 사용: Groq Qwen3 32B
# llm = init_chat_model("groq:qwen/qwen3-32b", temperature=0.3)

# 나중에 사용: OpenAI (유료)
# llm = init_chat_model("openai:gpt-4.1-mini", temperature=0.3)

llm

ChatOllama(output_version=None, model='gemma4:e2b', temperature=0.3)

In [4]:
# 가장 간단한 호출: 문자열 직접 전달
response = llm.invoke("LangChain v1.0 의 핵심 철학을 한 문장으로 설명하세요.")
print(response.content)

LangChain의 핵심 철학은 **거대 언어 모델(LLM)을 단순한 텍스트 생성 도구를 넘어, 외부 데이터, 다른 애플리케이션 및 도구와 연결하여 복잡하고 추론 기반의 워크플로우를 구축할 수 있도록 하는 것**입니다.


In [5]:
response

AIMessage(content='LangChain의 핵심 철학은 **거대 언어 모델(LLM)을 단순한 텍스트 생성 도구를 넘어, 외부 데이터, 다른 애플리케이션 및 도구와 연결하여 복잡하고 추론 기반의 워크플로우를 구축할 수 있도록 하는 것**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-07-22T06:32:23.897709Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4385444208, 'load_duration': 228827875, 'prompt_eval_count': 34, 'prompt_eval_duration': 78888000, 'eval_count': 393, 'eval_duration': 4075160000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019f8886-6f37-7b53-969f-7dca5cc435aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 393, 'total_tokens': 427})

## 3. Messages — 대화의 기본 단위

v1.0에서는 메시지 타입이 `langchain.messages` 모듈에 통합되어 있습니다. 메시지는 `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 등의 클래스로 표현되며, 모델 호출 시 이들의 리스트를 입력으로 사용합니다.

In [6]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="당신은 사내 IT 도우미입니다. 간결하게 답하세요."),
    HumanMessage(content="VPN 연결이 자꾸 끊기는데 어떻게 해야 하나요?"),
]
reply = llm.invoke(messages)
print(reply.content)
print("\n메시지 타입:", type(reply).__name__)  # AIMessage

다음과 같은 순서로 확인해 보세요.

1. **인터넷 연결 확인:** VPN 연결 전에 일반 인터넷 연결이 안정적인지 확인해 주세요.
2. **VPN 클라이언트 재시작:** 사용 중인 VPN 프로그램을 완전히 종료했다가 다시 시작해 보세요.
3. **장치 재부팅:** 컴퓨터나 연결 장치를 재부팅하여 일시적인 오류를 해결해 보세요.
4. **방화벽/보안 프로그램 확인:** 회사 방화벽이나 백신 프로그램이 VPN 연결을 차단하고 있지 않은지 확인해 보세요.

위 방법으로 해결되지 않으면, 사용 중인 **VPN 종류**와 **오류 메시지**를 알려주시면 더 정확한 도움을 드릴 수 있습니다.

메시지 타입: AIMessage


### AIMessage의 메타데이터 활용

`AIMessage` 객체에는 응답 본문 외에도 토큰 사용량, 모델 식별자, 종료 사유 등의 메타데이터가 포함되어 있습니다. 비용 추적과 디버깅에 유용합니다.

In [7]:
print("content:", reply.content[:80], "...")
print("usage_metadata:", reply.usage_metadata)  # 토큰 사용량
print("response_metadata.model:", reply.response_metadata.get("model_name"))

content: 다음과 같은 순서로 확인해 보세요.

1. **인터넷 연결 확인:** VPN 연결 전에 일반 인터넷 연결이 안정적인지 확인해 주세요.
2. ** ...
usage_metadata: {'input_tokens': 46, 'output_tokens': 432, 'total_tokens': 478}
response_metadata.model: gemma4:e2b


## 4. Prompt Template — 변수 기반 프롬프트

동일한 프롬프트 패턴을 여러 입력에 재사용할 때 사용합니다. `ChatPromptTemplate`은 시스템 메시지와 사용자 메시지의 템플릿을 정의하고, 변수 치환을 통해 일관된 형태의 메시지 리스트를 생성합니다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 입니다. 친절하고 전문적으로 답변하세요."),
    ("human", "{question}"),
])

# 변수 채우기
filled = prompt.invoke({
    "role": "ABC 사내 HR 도우미",
    "question": "연차 사용은 며칠 전까지 신청해야 하나요?",
})
print(filled)

messages=[SystemMessage(content='당신은 ABC 사내 HR 도우미 입니다. 친절하고 전문적으로 답변하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='연차 사용은 며칠 전까지 신청해야 하나요?', additional_kwargs={}, response_metadata={})]


In [9]:
for msg in filled.messages:
    msg.pretty_print()

================================ System Message ================================

당신은 ABC 사내 HR 도우미 입니다. 친절하고 전문적으로 답변하세요.
================================ Human Message =================================

연차 사용은 며칠 전까지 신청해야 하나요?


## 5. Output Parser — 응답 변환

`StrOutputParser`는 가장 단순한 형태의 파서로, `AIMessage` 객체에서 문자열 `content`만 추출합니다. 이후 절에서 다룰 LCEL 체인의 마지막 단계에 자주 배치되어, 후속 코드가 다루기 쉬운 일반 문자열을 반환하도록 합니다.

In [10]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
text = parser.invoke(reply)   # AIMessage → str
print(type(text), "\n", text[:80], "...")

<class 'langchain_core.messages.base.TextAccessor'> 
 다음과 같은 순서로 확인해 보세요.

1. **인터넷 연결 확인:** VPN 연결 전에 일반 인터넷 연결이 안정적인지 확인해 주세요.
2. ** ...


<!-- VIDEO: 2 / LCEL 파이프 — 한 줄로 끝나는 체인 -->

## 6. LCEL — 파이프 연산자 기반의 체인 구성

**LCEL(LangChain Expression Language)** 은 `|` 파이프 연산자를 사용해 컴포넌트를 연결하고, 체인을 **선언적으로 정의**하는 표현 방식입니다.

### 비유 1: 유닉스 파이프

`cat file | grep ERROR | wc -l`처럼, 왼쪽 컴포넌트의 출력이 오른쪽 컴포넌트의 입력으로 전달되는 단방향 흐름과 동일한 철학을 따릅니다.

```python
prompt | llm | parser
#  │       │      └── AIMessage → str (문자열 추출)
#  │       └────────── 채워진 프롬프트 → AIMessage (LLM 호출)
#  └────────────────── 변수 dict → 채워진 프롬프트 (Template 적용)
```

### 비유 2: 공정 라인

원자재(입력 변수) → 공정 1(프롬프트 템플릿 적용) → 공정 2(LLM 추론) → 공정 3(파서) → 결과물(문자열)의 흐름입니다. 각 공정은 독립적이므로 한 단계만 다른 컴포넌트로 교체해도 전체 라인은 그대로 동작합니다(예: `StrOutputParser`만 JSON 파서로 교체).

### 다이어그램

```mermaid
flowchart LR
    IN(["📥 dict input<br/>{role, question}"]):::input
    PT["📝 Prompt<br/>ChatPromptTemplate<br/>{role} · {question}"]:::node
    LM["🤖 LLM<br/>init_chat_model<br/>→ AIMessage"]:::node
    PA["🔍 Parser<br/>StrOutputParser<br/>→ str"]:::node
    OUT(["📤 최종 문자열"]):::output

    IN --> PT --> LM --> PA --> OUT

    classDef input  fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef node   fill:#1e293b,stroke:#475569,color:#e2e8f0
    classDef output fill:#059669,stroke:#047857,color:#fff
```

코드 표현은 다음과 같이 간결합니다.

```
prompt | llm | parser
```

### 파이프 연산자의 이점

1. **가독성**: "프롬프트 → 모델 → 파서"의 처리 흐름이 코드에 그대로 드러납니다.
2. **재사용성**: `prompt | llm` 부분을 변수로 분리하면 서로 다른 파서와 조합할 수 있습니다.
3. **공통 인터페이스**: 각 Runnable은 `.invoke()`, `.stream()`, `.batch()`를 자동 지원합니다.

In [11]:
chain = prompt | llm | parser

# 한 번의 invoke 로 prompt → llm → parser 전체가 실행됨
answer = chain.invoke({
    "role": "ABC HR 도우미",
    "question": "연차는 언제까지 신청해야 하나요?",
})
print(answer)

안녕하세요! ABC HR 도우미입니다. 연차 신청 시기에 대해 궁금하시군요.

연차(휴가) 신청 기한은 **회사마다 내부 규정 및 취업 규칙에 따라 다를 수 있습니다.** 따라서 가장 정확한 정보는 **소속 회사의 인사 규정**을 확인하시는 것이 가장 중요합니다.

하지만 일반적으로 많은 회사에서 권장하는 사항과 절차는 다음과 같습니다.

### 📌 확인해야 할 주요 사항

1. **회사 내부 규정 확인:**
   * 회사 내의 취업 규칙, 복무 규정 또는 사내 인트라넷(HR 시스템)에 연차 신청 및 사용에 관한 구체적인 지침이 명시되어 있습니다. 이곳을 먼저 확인해 주세요.

2. **상사와의 협의:**
   * 휴가를 계획하기 전에 반드시 직속 상사에게 필요한 기간과 일정을 논의하고 승인을 받는 것이 필수적입니다.

3. **신청 기한 준수:**
   * 보통 연차는 **사용하기 전에 미리 신청**해야 하며, 회사의 업무 운영에 지장이 없도록 적절한 시기에 요청하는 것이 좋습니다. (예: 최소 며칠 전 통보 등)

### ✅ 추천하는 행동 단계

가장 안전하고 정확하게 신청하시기 위해 다음 단계를 따라주세요.

1. **사내 시스템 확인:** 회사에서 사용하는 전자 결재 시스템이나 HR 포털에 로그인하여 연차 신청 절차와 마감일을 확인해 보세요.
2. **인사팀(HR) 문의:** 만약 내부 규정을 찾기 어렵다면, 직접 인사팀에 문의하시면 가장 정확한 답변을 얻으실 수 있습니다.

궁금한 점이 있다면 언제든지 다시 질문해 주세요. 친절하게 도와드리겠습니다! 😊


### 스트리밍 호출

LLM은 토큰 단위로 응답을 생성하므로, `.stream()`을 사용하면 토큰이 생성되는 즉시 클라이언트로 전달받을 수 있습니다. 사용자 응답 지연 체감을 줄이는 데 유용합니다.

In [12]:
for chunk in chain.stream({
    "role": "ABC IT 도우미",
    "question": "비밀번호 정책을 한 문단으로 알려주세요.",
}):
    print(chunk, end="", flush=True)
print()

안전한 비밀번호 정책은 사용자의 계정을 보호하기 위해 여러 요소를 결합하여 설정하는 것이 중요합니다. 기본적으로 비밀번호는 최소 10자리 이상으로 설정하고, 영문 대소문자, 숫자, 특수문자를 조합하여 복잡성을 높여야 합니다. 또한, 타인에게 쉽게 추측할 수 없는 개인적인 정보(생일, 이름 등)를 피하고, 주기적으로 비밀번호를 변경하여 보안 취약점을 최소화해야 하며, 가능한 경우 다중 인증(MFA)을 함께 사용하여 계정 보안 수준을 극대화하는 것을 권장합니다.


### 배치 호출

`.batch()`를 사용하면 여러 입력을 한 번의 호출 단위로 묶어 병렬 처리할 수 있습니다. 다수의 독립적인 입력을 처리할 때 전체 처리 시간을 단축할 수 있습니다.

In [13]:
results = chain.batch([
    {"role": "HR 도우미", "question": "경조사 휴가 일수를 알려주세요."},
    {"role": "IT 도우미", "question": "VPN 접속 방법은?"},
    {"role": "재무 도우미", "question": "출장비는 얼마까지 지원되나요?"},
])
for i, r in enumerate(results, 1):
    print(f"[{i}] {r[:60]}...")

[1] 안녕하세요! HR 도우미입니다. 경조사 휴가 일수에 대해 문의 주셨군요.

죄송하지만, **어떤 회사(또는 ...
[2] 안녕하세요! IT 도우미입니다. VPN 접속 방법에 대해 궁금하시군요. VPN은 인터넷 보안과 개인 정보 보...
[3] 안녕하세요! 재무 도우미로서 정확한 정보를 안내해 드리기 위해서는 **어떤 회사 또는 어떤 규정**에 대한 ...


### 배치 입력을 스트림으로 처리

요청별로 동일한 체인을 순차 실행하면서, 각 응답을 토큰이 생성되는 대로 출력합니다.


In [20]:
stream_inputs = [
    {"role": "HR 도우미", "question": "경조사 휴가 일수를 알려주세요."},
    {"role": "IT 도우미", "question": "VPN 접속 방법은?"},
    {"role": "재무 도우미", "question": "출장비는 얼마까지 지원되나요?"},
]

for idx, item in enumerate(stream_inputs, 1):
    print(f"[{idx}] {item['role']}")
    for chunk in chain.stream(item):
        print(chunk, end="", flush=True)
    print("\n" + "-" * 40)


[1] HR 도우미
안녕하세요! HR 도우미입니다. 경조사 휴가 일수에 대해 문의주셨군요.

죄송하지만, 저는 특정 회사나 조직의 내부 규정을 알 수 없기 때문에 **정확한 휴가 일수를 바로 알려드리기는 어렵습니다.**

경조사 휴가(경조사 휴가)는 **회사마다, 그리고 취업규칙이나 인사 규정에 따라 적용 기준과 일수가 모두 다릅니다.**

**정확한 정보를 확인하시려면 다음 방법들을 이용해 보시는 것을 추천드립니다:**

1. **회사 내부 규정 확인:** 회사 내의 취업규칙, 복무 규정 또는 사내 인트라넷(HR 시스템)에서 경조사 휴가 관련 규정을 확인해 보세요.
2. **인사팀(HR팀) 문의:** 가장 정확한 정보는 소속된 회사의 인사팀에 직접 문의하시는 것입니다.

혹시 회사에서 일반적으로 적용되는 기준이나, 제가 알고 있는 일반적인 정보에 대해 궁금하신 점이 있다면 다시 질문해 주세요. 친절하게 도와드리겠습니다! 😊
----------------------------------------
[2] IT 도우미
안녕하세요! IT 도우미입니다. VPN 접속 방법에 대해 질문해 주셨군요. VPN은 보안을 강화하고 인터넷에 더 안전하게 접속할 수 있도록 도와주는 매우 유용한 도구입니다.

하지만 **어떤 VPN 서비스(예: NordVPN, ExpressVPN, 또는 회사에서 제공하는 VPN 등)를 사용하시는지, 그리고 어떤 기기(PC, 스마트폰, 공유기 등)를 사용하시는지**에 따라 접속 방법이 조금씩 달라집니다.

가장 일반적인 VPN 접속 방법을 단계별로 안내해 드리겠습니다. 원하시는 상황에 맞는 방법을 확인해 보세요!

---

### 1. 상용 VPN 서비스 (앱/소프트웨어 이용 시)

대부분의 개인 사용자는 특정 VPN 회사의 공식 앱이나 소프트웨어를 통해 접속합니다. 이 방법이 가장 일반적이고 쉽습니다.

**✅ 접속 단계:**

1. **VPN 서비스 선택 및 구독:** 먼저 사용하고자 하는 VPN 서비스에 가입하고 구독을 완료해야 합니다.
2. 

## 7. Runnable 컴포넌트

LCEL 체인을 구성하는 모든 부품은 공통적으로 `Runnable` 인터페이스를 따릅니다. 보다 복잡한 처리 흐름을 구성할 때 다음 유틸리티 컴포넌트를 활용합니다.

| 컴포넌트 | 역할 |
|---|---|
| `RunnableSequence` | 파이프라인 (`\|` 연산자로 자동 생성) |
| `RunnableParallel` | 여러 체인을 병렬 실행 |
| `RunnablePassthrough` | 입력을 그대로 전달 (딕셔너리 필드 채울 때) |
| `RunnableLambda` | 일반 파이썬 함수를 Runnable로 래핑 |

### 7.1 `RunnablePassthrough` + `RunnableParallel`

동일한 입력에 대해 **여러 관점의 응답을 동시에** 생성하는 패턴입니다. 각 분기 체인이 독립적으로 실행되므로 직렬 호출 대비 응답 시간을 단축할 수 있습니다.

In [14]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

pro_prompt  = ChatPromptTemplate.from_messages([("system", "긍정적 관점으로 답하세요."), ("human", "{q}")])
con_prompt  = ChatPromptTemplate.from_messages([("system", "비판적 관점으로 답하세요."), ("human", "{q}")])

pros_chain = pro_prompt | llm | parser
cons_chain = con_prompt | llm | parser

debate = RunnableParallel(
    question=RunnablePassthrough(),   # 입력의 q를 그대로 통과
    pros=pros_chain,
    cons=cons_chain,
)

out = debate.invoke({"q": "재택근무를 전면 도입하는 것은 좋은가?"})
print("PROS:", out["pros"][:100], "...")
print("CONS:", out["cons"][:100], "...")

PROS: 재택근무의 전면 도입은 **많은 긍정적인 측면을 제공하며, 현대 사회와 업무 환경에 매우 적합한 발전적인 변화**라고 볼 수 있습니다. 이는 단순히 근무 장소를 바꾸는 것을 넘어, ...
CONS: 재택근무(Remote Work)의 전면 도입은 단순히 '좋다' 또는 '나쁘다'로 단정할 수 없는 복합적인 문제입니다. 이는 조직의 특성, 산업 분야, 팀 문화, 그리고 도입 방식에 ...


### 7.2 `RunnableLambda` — 파이썬 함수 삽입

체인 중간에 전처리 또는 후처리 단계를 추가할 때 사용합니다. 일반 파이썬 함수를 Runnable로 래핑해 LCEL 체인에 자연스럽게 결합할 수 있습니다.

In [15]:
from langchain_core.runnables import RunnableLambda

def shorten(text: str, max_len: int = 80) -> str:
    return text if len(text) <= max_len else text[:max_len] + " ..."

# 결과를 짧게 자르는 체인
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 요약하세요."),
    ("human", "{text}"),
])

chain2 = summary_prompt | llm | parser | RunnableLambda(shorten)
print(chain2.invoke({"text": "LangChain 은 LLM 기반 애플리케이션을 만들기 위한 프레임워크로, 프롬프트 관리, 메모리, 도구 호출, 에이전트 실행, RAG 등 여러 기능을 제공합니다."}))

LangChain은 프롬프트 관리, 메모리, 도구 호출, 에이전트 실행, RAG 등 다양한 기능을 제공하여 LLM 기반 애플리케이션을 구축하기  ...


## 8. 멀티턴 대화를 위한 메시지 누적

간단한 대화 흐름은 메시지 리스트를 직접 관리하는 방식으로 구현할 수 있습니다. 보다 복잡한 메모리 관리(체크포인팅, 세션 단위 상태 저장 등)는 LangGraph의 `MemorySaver` 체크포인터에서 별도로 다룹니다.

In [16]:
history = [SystemMessage(content="당신은 ABC IT 도우미입니다.")]

def chat(user_msg: str) -> str:
    history.append(HumanMessage(content=user_msg))
    ai = llm.invoke(history)
    history.append(ai)
    return ai.content

print("User:", "안녕, 내 이름은 지훈이야.")
print("Bot :", chat("안녕, 내 이름은 지훈이야."))
print()
print("User:", "내 이름을 기억해?")
print("Bot :", chat("내 이름을 기억해?"))

User: 안녕, 내 이름은 지훈이야.
Bot : 안녕하세요, 지훈님! 만나서 반갑습니다. 저는 ABC IT 도우미입니다. 😊

지훈님, 제가 어떤 도움을 드릴 수 있을까요? 궁금한 점이나 도움이 필요한 것이 있다면 언제든지 말씀해주세요!

User: 내 이름을 기억해?
Bot : 네, 기억하고 있습니다! 😊

지훈님, 앞으로 편하게 지훈님이라고 부르겠습니다.

혹시 지훈님께 필요한 IT 관련 정보나 다른 도움이 있으신가요?


<!-- VIDEO: 3 / 한국어 실전 LCEL — 메뉴 추천 + 뉴스 헤드라인 병렬 처리 -->

---

## 9. 한국어 비즈니스 시나리오 실습

### 9.1 메뉴 추천 체인

LCEL의 강점은 **여러 단계의 처리 흐름을 한 줄로 정리**할 수 있다는 점입니다. 사용자 취향 입력 → 메뉴 추천 → 가독성 있는 출력 형식까지 이어지는 체인을 구성합니다.

In [17]:
menu_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 한국 음식점 추천 전문가입니다. "
               "사용자의 기분·날씨·예산을 보고 가장 잘 어울리는 메뉴 3개를 추천하세요. "
               "각 메뉴마다 이모지 1개와 한 줄 추천 이유를 붙이세요."),
    ("human", "기분: {mood}\n날씨: {weather}\n예산: {budget}원"),
])

menu_chain = menu_prompt | llm | parser

print(menu_chain.invoke({"mood": "축 처지는 월요일", "weather": "비 오고 쌀쌀함", "budget": "12000"}))

축 처지는 월요일과 비 오는 쌀쌀한 날씨에는 몸을 따뜻하게 해주고 마음까지 위로해 주는 **뜨끈하고 얼큰한 국물 요리**가 최고입니다! 예산 범위 내에서 기분 전환이 될 만한 메뉴 3가지를 추천해 드립니다.

---

### 🍲 추천 메뉴 3가지

**1. 부대찌개 (Budae Jjigae)** 🌶️
*   **추천 이유:** 얼큰하고 푸짐한 재료들이 어우러져 스트레스를 날려버릴 만큼 만족감을 줍니다.

**2. 삼계탕 (Samgyetang)** 🐔
*   **추천 이유:** 쌀쌀한 날씨에 몸을 따뜻하게 데워주고 기력을 회복시켜주는 최고의 보양식입니다.

**3. 얼큰한 해물 칼국수 (Spicy Seafood Noodles)** 🍜
*   **추천 이유:** 뜨끈한 국물이 속을 편안하게 채워주며, 비 오는 날의 분위기와도 잘 어울립니다.


### 9.2 뉴스 헤드라인 — 요약과 감정 분석 병렬 처리

`RunnableParallel`을 활용해 요약 작업과 감정 분석 작업을 **동시에 실행**합니다. 두 작업을 직렬로 호출했을 때 대비 전체 응답 시간을 줄일 수 있습니다.

In [18]:
news_text = """
오늘 코스피 지수가 개장 직후 1.2% 급락하며 약세를 보였다. 외국인 투자자 순매도가 5일 연속 이어지면서 시장의 우려가 커지고 있다.
반면 일부 반도체 관련주는 AI 수요 기대감으로 강세를 보였다. 전문가들은 단기 변동성에 대비한 분산 투자를 권고하고 있다.
"""

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 뉴스를 정확히 3줄로 요약하세요. 각 줄은 핵심 사실 1개만 담습니다."),
    ("human", "{news}"),
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 뉴스의 시장 감정을 한 단어로 분류: 긍정/부정/중립. 그리고 그 이유를 한 문장으로."),
    ("human", "{news}"),
])

# 두 체인을 병렬 실행
analyze = RunnableParallel(
    summary=summary_prompt | llm | parser,
    sentiment=sentiment_prompt | llm | parser,
)

result = analyze.invoke({"news": news_text})
print("📋 [요약]")
print(result["summary"])
print("\n📊 [감정 분석]")
print(result["sentiment"])

📋 [요약]
코스피 지수가 개장 직후 1.2% 급락하며 약세를 보였습니다.
외국인 투자자의 순매도가 5일 연속 이어지면서 시장의 우려가 커지고 있습니다.
반면 일부 반도체 관련주는 AI 수요 기대감으로 강세를 보였으며, 전문가들은 분산 투자를 권고했습니다.

📊 [감정 분석]
**부정**

외국인 투자자들의 지속적인 순매도와 코스피 지수의 급락이 시장의 우려를 키우고 있기 때문입니다.


---

## 실습 과제: 번역 + 톤 변환 체인

**요구사항**

1. `ChatPromptTemplate`으로 "입력된 한국어 문장을 영어로 번역한 뒤, 지정된 톤({tone}: formal/casual/polite)으로 다듬으세요"라는 프롬프트를 정의합니다.
2. `prompt | llm | parser` 체인으로 실행합니다.
3. 동일 문장을 세 가지 톤으로 `batch` 호출하여 결과를 비교합니다.

In [19]:
# 여기에 코드를 작성하세요
# Hint: ChatPromptTemplate.from_messages + chain = prompt | llm | parser + chain.batch([...])


<details>
<summary>모범 답안 보기</summary>

```python
tone_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the Korean sentence to English and rewrite it in a {tone} tone. Output English only."),
    ("human", "{text}"),
])

chain = tone_prompt | llm | parser

ko = "회의에 못 갈 것 같아요. 미안합니다."
results = chain.batch([
    {"text": ko, "tone": "formal"},
    {"text": ko, "tone": "casual"},
    {"text": ko, "tone": "polite"},
])
for tone, r in zip(["formal", "casual", "polite"], results):
    print(f"[{tone}] {r}")
```

</details>

---

## 트러블슈팅

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `ImportError: init_chat_model` | LangChain v0.x 사용 중 | `pip install -U langchain langchain-core` |
| `ModuleNotFoundError: langchain_google_genai` | 통합 패키지 미설치 | `pip install langchain-google-genai` |
| `RateLimitError: 429` (Gemini 사용) | 분당 15회 한도 초과 | 30초 대기 또는 Groq로 전환 |
| `chain.batch`의 응답이 느림 | 동기 실행 사용 중 | `chain.abatch()` + `asyncio.run()` 으로 비동기 처리 |
| `RunnableParallel` 결과가 dict가 아님 | 입력이 dict 형식이 아님 | `{"key": value}` 형식으로 `invoke` 호출 |

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- `init_chat_model("provider:model")`을 통한 모델 통합 초기화
- Messages → Prompt Template → Output Parser의 표준 처리 흐름
- LCEL `|` 파이프 연산자를 통한 체인 선언
- `.invoke()`, `.stream()`, `.batch()`의 공통 지원
- `RunnableParallel`을 통한 병렬 실행, `RunnableLambda`를 통한 함수 삽입
- 한국어 비즈니스 시나리오(메뉴 추천, 뉴스 분석) 적용 사례

### 다음 노트북(03번) 예고

본 노트북에서 사용한 프롬프트는 단일 텍스트 형태였습니다. 실무에서는 다음과 같은 정밀한 통제가 필요합니다.

- "이러한 예시처럼 응답해 달라" — **Few-shot 프롬프팅**
- "정해진 JSON 형식으로 응답해 달라" — **Pydantic 기반 구조화 출력**

다음 노트북에서는 LLM 응답의 패턴과 형식을 정밀하게 제어하는 방법을 다룹니다.